# 04. 네 가지 실패 모드

Poisoning, Distraction, Confusion, Clash는 입력에서 생기는 문제를 서로 다른 관점으로 설명합니다. 실험에서는 자극의 이름과 실제 출력 관찰을 구분합니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=300)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 다섯 가지 입력 조건을 만듭니다

기준 입력을 유지하면서 잘못된 메모, 무관한 정보, 비슷한 기관, 상충 규정을 하나씩 추가합니다.

In [3]:
clean = "[E01 p.2] 미르시립도서관은 매주 월요일 휴관입니다."
noise = "회의 안건: 행사 배너와 좌석 배치를 논의했습니다. " * 12
conditions = {
    "clean": clean,
    "poisoning": f"{clean}\n[미검증 메모] 휴관일은 화요일입니다.",
    "distraction": f"{noise}\n{clean}",
    "confusion": f"{clean}\n[미르고등학교 도서실] 화요일 휴관입니다.",
    "clash": f"{clean}\n[합성 부록] 미르시립도서관은 수요일 휴관입니다.",
}
print("실험 조건:", list(conditions))

실험 조건: ['clean', 'poisoning', 'distraction', 'confusion', 'clash']


## 3. 같은 질문으로 다섯 조건을 실행합니다

어떤 실패가 나타날지 미리 정하지 않습니다. 조건 이름, 실제 입력 토큰, 응답 원문을 나란히 기록합니다.

In [4]:
results = {}
for name, context in conditions.items():
    messages = [
        ("system", "근거만 사용하고, 충돌하면 확인이 필요하다고 답하세요."),
        ("user", f"근거:\n{context}\n\n질문: 정기 휴관일은 언제인가요?"),
    ]
    response = model.invoke(messages)
    usage = response.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    results[name] = response
    print(f"\n[{name}] input_tokens={usage['input_tokens']}")
    print(response.text)


[clean] input_tokens=65
정기 휴관일은 **매주 월요일**입니다.



[poisoning] input_tokens=80
근거에 따르면 **미르시립도서관의 정기 휴관일은 매주 월요일**입니다.  
다만 메모에는 화요일이라고 되어 있어 **충돌하므로 추가 확인이 필요합니다**.



[distraction] input_tokens=269
정기 휴관일은 **매주 월요일**입니다.



[confusion] input_tokens=80
근거가 서로 충돌합니다.  
- 미르시립도서관: **매주 월요일 휴관**
- 미르고등학교 도서실: **화요일 휴관**

따라서 **정기 휴관일은 확인이 필요합니다.**



[clash] input_tokens=85
근거상 **정기 휴관일은 월요일과 수요일**입니다.  
다만 서로 다른 근거가 각각 제시되어 있으므로, 실제 운영 기준은 **확인이 필요합니다**.


## 4. 출력에서 관찰한 문자열을 비교합니다

요일과 확인 요청 표현의 등장 여부만 표시합니다. 이 표만으로 답의 완결성을 확정하지 않고 응답 원문을 함께 읽습니다.

In [5]:
for name, response in results.items():
    answer = response.text
    observed = [day for day in ("월요일", "화요일", "수요일") if day in answer]
    if "확인" in answer or "충돌" in answer or "상충" in answer:
        observed.append("추가 확인 표현")
    print(name, observed or ["요일 표현 없음"])

clean ['월요일']
poisoning ['월요일', '화요일', '추가 확인 표현']
distraction ['월요일']
confusion ['월요일', '화요일', '추가 확인 표현']
clash ['월요일', '수요일', '추가 확인 표현']


## 정리

실패 자극을 넣었다는 사실만으로 실패가 발생했다고 기록하지 않습니다. 출력에서 관찰한 문장과 입력 근거를 연결합니다.

## 실패 재현: 상충 규정의 시행일을 기록하지 않습니다

두 규정에 시행일이 없으면 어느 규정이 질문 시점에 유효한지 판정할 수 없습니다.

In [6]:
def require_versions(items: list[str]) -> None:
    if any("시행일:" not in item for item in items):
        raise ValueError("상충 규정의 시행일이 없습니다.")

try:
    require_versions([conditions["clean"], conditions["clash"]])
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("시행일 누락 오류가 발생해야 합니다.")

관찰한 실패: 상충 규정의 시행일이 없습니다.


## 수정: 각 규정에 시행일을 기록합니다

시행일·요일·출처를 구조화하고 질문 기준일에 유효한 규정 가운데 가장 최근 규정을 선택합니다.

In [7]:
versioned = [
    {"effective_date": "2025-01-01", "weekday": "월요일", "source": "E01 p.2"},
    {"effective_date": "2026-01-01", "weekday": "수요일", "source": "합성 부록"},
]
as_of = "2026-09-08"
eligible = [record for record in versioned if record["effective_date"] <= as_of]
selected_rule = max(eligible, key=lambda record: record["effective_date"])
assert selected_rule["weekday"] == "수요일"
assert selected_rule["source"] == "합성 부록"
print("수정 후 적용 규정:", selected_rule)

수정 후 적용 규정: {'effective_date': '2026-01-01', 'weekday': '수요일', 'source': '합성 부록'}
